# NaverCafe Weekly Brand Summary
Generates weekly GPT-4 summaries of community discussions per brand,
highlighting new insights not covered in the previous 4 weeks.

**Run schedule:** Every Tuesday

**Pipeline steps:**
1. Verify today is Tuesday (scheduled run)
2. Fetch posts from the previous Sunday to Saturday
3. Generate weekly summary per brand using GPT-4
4. Upload summaries to data warehouse

In [ ]:
# pip install torch azure-storage-blob snowflake-connector-python snowflake-sqlalchemy==1.5.1 openai==0.28

In [ ]:
import os
import datetime
import pandas as pd
import openai
import snowflake.connector
from datetime import timedelta
from snowflake.sqlalchemy import URL
from sqlalchemy import create_engine
from snowflake.connector.pandas_tools import pd_writer

# Current time in KST (UTC+9)
now = datetime.datetime.now() + timedelta(hours=9)

# This pipeline is scheduled to run on Tuesdays only
if now.weekday() != 1:
    print('Not Tuesday. Skipping run.')
    exit()

# Set date range: previous Sunday to Saturday
last_sunday = now - timedelta(days=now.weekday() + 8)
last_saturday = now - timedelta(days=now.weekday() + 2)
start_date = last_sunday.strftime('%Y-%m-%d')
end_date = last_saturday.strftime('%Y-%m-%d')
print(f'Processing period: {start_date} to {end_date}')

In [ ]:
# ── DATABASE CONNECTION ──────────────────────────────────────────────────
# Set environment variables before running:
#   SNOWFLAKE_USER, SNOWFLAKE_PASSWORD, SNOWFLAKE_ACCOUNT
#   SNOWFLAKE_DATABASE, SNOWFLAKE_SCHEMA, SNOWFLAKE_WAREHOUSE
#   OPENAI_API_KEY

con = snowflake.connector.connect(
    user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'],
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    database=os.environ['SNOWFLAKE_DATABASE'],
    schema=os.environ['SNOWFLAKE_SCHEMA'],
    warehouse=os.environ['SNOWFLAKE_WAREHOUSE']
)
cur = con.cursor()

In [ ]:
# ── FETCH POSTS FOR THE TARGET WEEK ──────────────────────────────────────

content_query = f"""
SELECT *
FROM NLP_TABLE
WHERE WDATE BETWEEN '{start_date}' AND '{end_date}'
AND NOT (
   BOARD_PATH IN ('Deal Board')
   OR BOARD_PATH LIKE '%HotDeal%'
   OR TITLE LIKE '%postpartum care%'
   OR TITLE LIKE '%baby products%'
   OR TITLE LIKE '%review%'
   OR TITLE LIKE '%hot deal%'
);
"""

cur.execute(content_query)
content_columns = [desc[0] for desc in cur.description]
content = pd.DataFrame(cur.fetchall(), columns=content_columns)
content['CONTENTS'] = content['CONTENTS'].fillna('')
content['WDATE'] = pd.to_datetime(content['WDATE'])

# Add week start (Sunday-based) for grouping
content['PERIOD_START'] = content['WDATE'] - pd.to_timedelta(
    (content['WDATE'].dt.weekday + 1) % 7, unit='d'
)

In [ ]:
# ── GPT-4 WEEKLY SUMMARY ─────────────────────────────────────────────────
# Summarizes new insights from the current week compared to the past 4 weeks

openai.api_key = os.environ['OPENAI_API_KEY']
GPT_MODEL = 'gpt-4o-2024-08-06'

TARGET_BRANDS = ['Pampers', 'Huggies', 'Kindoh', 'Penelope', 'Mamipoko', 'Bosomi']

def gpt_summary(combined_contents, past_contents, brand):
    """Generate a concise summary of new brand-related insights for the week."""
    prompt = f"""
    As a marketer for {brand}, review this week's community posts and identify
    new insights not covered in the past 4 weeks. Summary must be in Korean, under 200 characters.

    Past 4 weeks:
    {past_contents}

    This week's posts:
    {combined_contents}

    Focus on what is new or significantly different about {brand} specifically.
    Exclude mentions of other brands.
    """
    messages = [
        {'role': 'system', 'content': f'You are a marketer for {brand} analyzing customer feedback.'},
        {'role': 'user', 'content': prompt}
    ]
    response = openai.ChatCompletion.create(model=GPT_MODEL, messages=messages)
    return response['choices'][0]['message']['content']

In [ ]:
# ── GENERATE WEEKLY SUMMARIES PER BRAND ──────────────────────────────────

summary_results = []

for brand in TARGET_BRANDS:
    print(f'Processing: {brand}')
    brand_content = content[content['BRAND'] == brand]
    week_start = brand_content['PERIOD_START'].min()

    while week_start <= brand_content['WDATE'].max():
        week_end = week_start + timedelta(days=6)
        week_data = brand_content[
            (brand_content['WDATE'] >= week_start) &
            (brand_content['WDATE'] <= week_end)
        ]
        if not week_data.empty:
            # Use top half of posts by view count for summary
            top_posts = week_data.sort_values('VIEWS', ascending=False).head(len(week_data) // 2)
            combined_contents = ' '.join(top_posts['CONTENTS'].tolist())
            past_contents = ''  # Can be populated from previous summaries if available

            summary_text = gpt_summary(combined_contents, past_contents, brand)
            summary_results.append({
                'BRAND': brand,
                'START_DATE': week_start.strftime('%Y-%m-%d'),
                'END_DATE': week_end.strftime('%Y-%m-%d'),
                'SUMMARY': summary_text
            })
        week_start += timedelta(days=7)

summary_df = pd.DataFrame(summary_results)
print(summary_df.head())

In [ ]:
# ── UPLOAD TO DATA WAREHOUSE ─────────────────────────────────────────────

engine = create_engine(URL(
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'],
    database=os.environ['SNOWFLAKE_DATABASE'],
    schema=os.environ['SNOWFLAKE_SCHEMA'],
    warehouse=os.environ['SNOWFLAKE_WAREHOUSE'],
))

table_name = 'SUMMARY_TABLE'

with engine.connect() as con:
    summary_df.to_sql(name=table_name, con=con, if_exists='append',
                      method=pd_writer, index=False)

print(f'Uploaded {len(summary_df)} summaries to {table_name}')